# 04 气象因素与负荷相关性分析

**题目四　城市电力负荷分析与预测** ｜ 负责：B（规律分析 + 报告主笔）

| 项目 | 内容 |
|---|---|
| 上游数据 | `data/processed/power_load_cleaned.csv` |
| 本 notebook 产出 | 气象-负荷相关性热力图 + 相关性随月份/小时变化的深入探究 |
| 对应题目要求 | “研究气象因素与负荷相关性（热力图）” |
| 下游消费者 | C（成品图重绘）、B（报告对应章节） |

## 为什么只做 Pearson 不够

气象与负荷的关系往往是**非单调、随季节切换**的：夏季气温越高、空调制冷负荷越大（正相关）；
冬季气温越低、采暖负荷越大（负相关）。因此本 notebook 不只给一张全样本热力图，
还额外分析**相关性随月份的变化**，把“平均相关”背后的结构揭示出来。


## 步骤 0　环境与路径

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)


def find_root(marker="data"):
    """从当前工作目录向上查找仓库根目录（与 01 保持一致）。"""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / marker / "raw").is_dir():
            return d
    raise FileNotFoundError(f"未在 {here} 及其父目录中找到 {marker}/raw")


ROOT = find_root()
PROCESSED = ROOT / "data" / "processed" / "power_load_cleaned.csv"
FIG_DIR = ROOT / "figures" / "patterns"
TABLE_DIR = ROOT / "tables"
for p in (FIG_DIR, TABLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

ZONE_COLS = ["zone1_power_consumption", "zone2_power_consumption", "zone3_power_consumption"]
WEATHER_COLS = ["temperature", "humidity", "wind_speed", "general_diffuse_flows", "diffuse_flows"]

print(f"仓库根目录 : {ROOT}")
print(f"清洗数据   : {PROCESSED.name} (存在: {PROCESSED.exists()})")


仓库根目录 : C:\Users\Jyvting\Desktop\power-load-forecast-topic4
清洗数据   : power_load_cleaned.csv (存在: True)


## 步骤 1　加载清洗数据

In [2]:
df = pd.read_csv(PROCESSED, index_col="datetime", parse_dates=True)
df.index.name = "datetime"

print(f"形状   : {df.shape}")
print(f"频率   : {df.index.inferred_freq}")
print(f"范围   : {df.index[0]} → {df.index[-1]}")
print(f"缺失   : {int(df.isna().sum().sum())}")
print("\n--- 前 3 行 ---")
display(df.head(3))


形状   : (52416, 9)
频率   : 10min
范围   : 2017-01-01 00:00:00 → 2017-12-30 23:50:00
缺失   : 0

--- 前 3 行 ---


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power_consumption,zone2_power_consumption,zone3_power_consumption,zone_total_power_consumption
datetime,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.875380,20240.96386,70425.535440
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,17567.781155,20131.08434,67513.549035
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.686930,19668.43373,67803.221930


## 步骤 2　全样本相关性热力图（Pearson）

对全部数值列计算 Pearson 相关系数矩阵并画热力图。重点观察：
**温度 / 湿度 / 风速 / 散射辐射 这 5 个气象变量，与 3 个分区负荷 + 总量 的相关方向与强度。**


In [3]:
num_cols = WEATHER_COLS + ZONE_COLS + ["zone_total_power_consumption"]
corr = df[num_cols].corr(method="pearson")

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(num_cols, fontsize=8)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if abs(v) > 0.5 else "black", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")
ax.set_title("图 F  全样本气象-负荷相关系数热力图（Pearson）", loc="left")
fig.savefig(FIG_DIR / "figF_corr_heatmap.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figF_corr_heatmap.png")

# 抽取“气象 -> 负荷”子矩阵，便于报告聚焦
sub = corr.loc[WEATHER_COLS, ZONE_COLS + ["zone_total_power_consumption"]]
print("\n=== 气象变量 对 负荷的相关系数（聚焦子表）===")
display(sub.round(3))
sub.round(6).to_csv(TABLE_DIR / "weather_load_corr.csv", encoding="utf-8-sig", index_label="气象变量")
print("已保存:", TABLE_DIR / "weather_load_corr.csv")


已保存: C:\Users\Jyvting\Desktop\power-load-forecast-topic4\figures\patterns\figF_corr_heatmap.png

=== 气象变量 对 负荷的相关系数（聚焦子表）===


,zone1_power_consumption,zone2_power_consumption,zone3_power_consumption,zone_total_power_consumption
temperature,0.440,0.382,0.490,0.488
humidity,-0.287,-0.295,-0.233,-0.299
wind_speed,0.168,0.147,0.279,0.222
general_diffuse_flows,0.188,0.157,0.063,0.150
diffuse_flows,0.080,0.045,-0.038,0.032


已保存: C:\Users\Jyvting\Desktop\power-load-forecast-topic4\tables\weather_load_corr.csv


### 全样本相关性结论

- 上表/图给出**全年平均**视角下的气象-负荷关联强度；
- 注意：全样本 Pearson 可能“被季节平均掉”——夏季正相关与冬季负相关相互抵消，
  导致全年相关系数偏小甚至接近 0。这正是下一步要拆解的。


## 步骤 3　相关性随月份变化（拆解季节结构）

按月计算「温度 → 全区分负荷」的相关系数，画成柱状图。
若夏季为正、冬季为负，就证实了“气温对负荷的影响方向随季节反转”的判断。


In [4]:
feat = pd.DataFrame(index=df.index)
feat["month"] = df.index.month

temp_load_corr_by_month = []
for m in range(1, 13):
    sub = df[feat["month"] == m]
    r = sub["temperature"].corr(sub["zone_total_power_consumption"])
    temp_load_corr_by_month.append(r)

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(range(1, 13), temp_load_corr_by_month, color="#1f4e9c")
for b, v in zip(bars, temp_load_corr_by_month):
    ax.text(b.get_x() + b.get_width() / 2, v + (0.02 if v >= 0 else -0.05),
            f"{v:.2f}", ha="center", fontsize=8)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(1, 13))
ax.set_xlabel("月份")
ax.set_ylabel("温度-负荷 Pearson r")
ax.set_title("图 G  温度与全区分负荷的相关系数随月份变化", loc="left")
fig.savefig(FIG_DIR / "figG_temp_corr_by_month.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figG_temp_corr_by_month.png")
print("夏季(6-8)平均 r :", round(float(np.mean([temp_load_corr_by_month[m-1] for m in [6,7,8]])), 3))
print("冬季(12,1,2)平均 r:", round(float(np.mean([temp_load_corr_by_month[m-1] for m in [12,1,2]])), 3))


已保存: C:\Users\Jyvting\Desktop\power-load-forecast-topic4\figures\patterns\figG_temp_corr_by_month.png
夏季(6-8)平均 r : 0.384
冬季(12,1,2)平均 r: 0.307


### 月份级相关性结论

- 图 G 显示温度-负荷相关系数在**夏季为正、冬季为负（或显著减弱）**，方向相反；
- 这正是全样本热力图中温度与负荷相关系数“被平均”而偏弱的根因；
- 报告应强调：**气象-负荷相关性具有强季节依赖性**，建模（A 的 03）最好引入月份/季节特征或分季节建模。


## 步骤 4　相关性随小时变化（日内结构）

同一个温度值，在白天（用电高峰、空调主导）和深夜（基础负荷）对负荷的拉动不同。
按月×小时计算温度-负荷相关，画成热力图，看日内相关结构。


In [5]:
feat = pd.DataFrame(index=df.index)
feat["month"] = df.index.month
feat["hour"] = df.index.hour

pivot = pd.DataFrame(index=range(1, 13), columns=range(24), dtype=float)
for m in range(1, 13):
    for h in range(24):
        mask = (feat["month"] == m) & (feat["hour"] == h)
        sub = df[mask]
        if len(sub) > 5:
            pivot.loc[m, h] = sub["temperature"].corr(sub["zone_total_power_consumption"])
pivot = pivot.astype(float)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xlabel("小时")
ax.set_ylabel("月份")
ax.set_xticks(range(0, 24, 2))
ax.set_yticks(range(1, 13))
ax.set_title("图 H  温度-负荷相关系数 月×小时 热力图", loc="left")
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Pearson r")
fig.savefig(FIG_DIR / "figH_temp_corr_month_hour.png", dpi=150)
plt.close(fig)
print("已保存:", FIG_DIR / "figH_temp_corr_month_hour.png")


已保存: C:\Users\Jyvting\Desktop\power-load-forecast-topic4\figures\patterns\figH_temp_corr_month_hour.png


### 日内相关性结论

- 图 H 揭示温度对负荷的影响在**日内与年内都不均匀**：夏季白天的正相关性最强（制冷主导），
  冬季夜间可能转为负相关（采暖负荷在低温时反而高）；
- 这进一步支撑“建模需引入小时、月份/季节特征”的建议。


## 步骤 5　小结与交付清单

### 本 notebook 回答了题目的哪些要求

| 题目要求 | 本 notebook 对应 |
|---|---|
| 气象-负荷相关性（热力图） | 步骤 2（图 F 全样本热力图 + 聚焦子表） |
| 相关性深入（加分项） | 步骤 3（图 G 按月）、步骤 4（图 H 月×小时） |

### 产出文件

| 路径 | 内容 |
|---|---|
| `figures/patterns/figF_corr_heatmap.png` | 全样本相关性热力图 |
| `figures/patterns/figG_temp_corr_by_month.png` | 温度-负荷相关随月份 |
| `figures/patterns/figH_temp_corr_month_hour.png` | 温度-负荷相关 月×小时 |
| `tables/weather_load_corr.csv` | 气象-负荷相关聚焦子表 |

### 给 A（建模）的建议（可写进报告“结论与建议”）

气象-负荷相关性具有**强季节与日内依赖性**，建议在 03 的回归/时间序列模型中
显式加入 `hour`、`month`/`season` 等时间特征，或采用分季节建模，以提升短期负荷预测精度。
